# RLSF omega selection: the dev-slice best-of-N pool

---
## 1 — Setup

In [ ]:
# 7B bf16 is ~15 GB of weights and the COMET encoder shares the card: 24 GB is comfortable.
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from pathlib import Path

# %cd into a repo that is already the working directory clones a second copy underneath it,
# so the guard is manage.py, not the directory name.
if not Path('manage.py').exists():
    if not Path('Style-Aware-MT/manage.py').exists():
        !git clone --branch feat/rlsf-implementation https://github.com/prnamhr/Style-Aware-MT.git
    %cd Style-Aware-MT
!git pull --ff-only
!git rev-parse --short HEAD

In [ ]:
import subprocess
import sys

# The one interpreter every shell cell below runs through. `python3` on a rented host is a
# different install from the kernel, and the two stacks drift the moment either is upgraded.
PY = sys.executable
COMET_PY = '.venv-comet/bin/python'
print('kernel', PY)

In [ ]:
# %pip installs into the kernel; !pip may not.
%pip install -q -r requirements.txt

In [ ]:
import numpy
import transformers

# COMET gets its own interpreter. Installing requirements-comet.txt into the kernel downgrades
# transformers and numpy under the generator, which then runs on a stack nothing else uses.
if not Path(COMET_PY).exists():
    pip = [COMET_PY, '-m', 'pip', 'install', '-q']
    subprocess.run([PY, '-m', 'venv', '.venv-comet'], check=True)
    subprocess.run([*pip, '--upgrade', 'pip'], check=True)
    subprocess.run([*pip, 'setuptools<81'], check=True)
    subprocess.run([*pip, '-r', 'requirements-comet.txt'], check=True)
subprocess.run([COMET_PY, '-c', 'import comet; print("comet ok")'], check=True)

print(f'kernel transformers {transformers.__version__}, numpy {numpy.__version__}')
assert transformers.__version__.startswith('5.'), "COMET's pins landed in the kernel"
assert numpy.__version__.startswith('2.'), "COMET's pins landed in the kernel"

In [ ]:
import getpass
import logging
import os

# Set here rather than in a shell cell: the CLI runs and the Kiwi worker are children of this
# kernel and inherit os.environ, so this is the one place the keys have to exist.
for var in ('OPENAI_API_KEY', 'HF_TOKEN'):
    if not os.environ.get(var):
        os.environ[var] = getpass.getpass(f'{var}: ')
logging.getLogger('httpx').setLevel(logging.WARNING)
print({var: bool(os.environ.get(var)) for var in ('OPENAI_API_KEY', 'HF_TOKEN')})

In [ ]:
# wmt22-cometkiwi-da is gated, and the worker is what needs the access; failing here beats
# failing an hour into the sampling.
check = '''
import os
from huggingface_hub import HfApi, __version__
api, tok = HfApi(), os.environ.get("HF_TOKEN")
print("hub", __version__, "| whoami:", api.whoami(token=tok)["name"])
api.list_repo_files("Unbabel/wmt22-cometkiwi-da", token=tok)
print("cometkiwi access OK")
'''
subprocess.run([COMET_PY, '-c', check], check=True)

---
## 2 — Pre-flight

In [ ]:
import hashlib
import json
import math

import yaml

from src.rlsf.config import judge_concurrency, load_config, make_judge_client, reward_config
from src.rlsf.pool import measured_per_call_usd, plan, pool_settings, read_dev
from src.rlsf.reward import load_train_template

CONFIG = 'configs/rlsf.yaml'

# The caps were declared 2026-08-08, so this loads with require_caps on.
cfg = load_config(CONFIG)
settings = pool_settings(cfg)
N = settings['n']
G = cfg['rlsf']['rollout']['group_size']
sources, refs = read_dev(cfg['data']['dev_file'])

# -- the pool's own ceiling, separate from the training caps: this notebook may spend
#    pool.judge_calls and no more, whatever caps.max_judge_calls allows
assert len(sources) * N == settings['judge_calls'], (
    f"{len(sources)} segments x N={N} is {len(sources) * N} calls against a declared "
    f"ceiling of {settings['judge_calls']}"
)
assert settings['judge_calls'] <= cfg['rlsf']['caps']['max_judge_calls']
assert N <= cfg['rlsf']['caps']['group_size_ceiling']

# -- the locked control: a quantized or swapped base is a different experiment
gen = cfg['generator']
assert gen['model'] == 'Qwen/Qwen2.5-7B-Instruct', gen['model']
assert gen['load_in_4bit'] is False, 'quantizing redefines the frozen base'
assert Path(gen['adapter_path'], 'adapter_config.json').exists(), (
    f"{gen['adapter_path']} is missing; copy the frozen PEFT checkpoint onto this host first"
)

# -- greedy sampling gives a group zero variance and the whole pool is uninformative
assert cfg['rlsf']['rollout']['temperature'] > 0, 'greedy samples cannot be ranked'

# -- reward_config rescales to unit ||omega||, so these are not the raw config numbers
w = reward_config(cfg).weights
print(f"policy {gen['model']} + {gen['adapter_path']}")
print(f"pool   {len(sources)} dev segments x N={N} at T={cfg['rlsf']['rollout']['temperature']}")
print('reward', {k: round(v, 3) for k, v in w.items()},
      f"||omega|| = {math.hypot(*w.values()):.3f}")

In [ ]:
# -- the reward judge must not be either evaluation rater, or the pool spends a rater on
#    the one condition that most needs a rater it was not trained against
raters = {yaml.safe_load(Path(p).read_text())['judge']['model']
          for p in ('configs/judge_eval.yaml', 'configs/judge_eval_gpt.yaml')}
assert cfg['judge']['model'] not in raters, (cfg['judge']['model'], raters)

# -- seeded, because under group normalization a rater flipping a 3 to a 4 inverts an
#    advantage sign, and this pool is re-ranked four times off one set of verdicts
assert cfg['judge']['temperature'] == 0.0 and cfg['judge']['seed'] == 42

# -- the rubric must be the frozen one; load_train_template raises on drift, this reports it
text = load_train_template()
digest = hashlib.sha256(text.encode()).hexdigest()
frozen = json.loads(Path('prompts/hashes.json').read_text())['templates']
assert digest == frozen['judge_train.txt']['sha256']
assert cfg['template_file'] == 'prompts/judge_train.txt', 'the eval rubric would be circular'

print(f"reward judge {cfg['judge']['model']}, distinct from {sorted(raters)}")
print(f"rubric verified {digest[:16]}")

In [ ]:
# -- the dev slice, against the manifest written when it was carved
man = json.loads(Path('data/splits/rlsf_dev_manifest.json').read_text())
for name, want in man['hashes'].items():
    got = hashlib.sha256((Path('data/splits') / name).read_bytes()).hexdigest()
    assert got == want, f'{name} differs from the manifest'
print(f"dev slice {man['counts']['rlsf_dev']} segments, {man['counts']['dev_works']} works")

# -- the slice is not unseen by the model; it selects weights, it does not measure them
print('\n'.join('  ' + c for c in man['caveats']))

In [ ]:
# The rate is measured, not assumed: token counts from the smoke priced at whatever the
# client charges for this model today. docs/budget.md quotes $0.29 for the whole pool.
judge_client = make_judge_client(cfg)
rate = measured_per_call_usd(judge_client, cfg['judge']['model'])
p = plan(len(sources), N, rate)

print(f"{p['samples']} completions, {p['judge_calls']} judge calls at concurrency "
      f"{judge_concurrency(cfg)}")
print(f"${rate:.3e} per call -> ${p['est_usd']:.2f}  (docs/budget.md: $0.29)")
print(f"pool ceiling {settings['judge_calls']} calls; "
      f"caps {cfg['rlsf']['caps']['max_judge_calls']} calls / "
      f"${cfg['rlsf']['caps']['max_judge_spend_usd']}")

---
## 3 — Free pass

In [ ]:
!{PY} manage.py rlsf_pool --config {CONFIG} --segments 2 --skip_judge \
    --out outputs/rlsf/pool_free.jsonl

---
## 4 — Paid pass

In [ ]:
!{PY} manage.py rlsf_pool --config {CONFIG} --resume --yes

---
## 5 — Read the pool

In [ ]:
from src.rlsf.pool import read_pool, sidecar

POOL = Path(cfg['output']['pool'])
rows = read_pool(POOL)
sizes = {len(r['hyps']) for r in rows}
unmeasured = sum(v is None or v != v for r in rows for v in r['scores']['judge'])

print(f"{POOL}: {len(rows)} segments, {sorted(sizes)} completions each, "
      f"{POOL.stat().st_size / 1e6:.1f} MB")
assert sizes == {N} and len(rows) == len(sources), 'the pool is incomplete'
print(f"{unmeasured} samples have no judge verdict "
      f"({unmeasured / (len(rows) * N):.2%}); those are dropped, not scored 1")

In [ ]:
# The measured rate this pool paid, against what section 2 planned and what budget.md holds.
u = json.loads(sidecar(POOL, 'usage.json').read_text())
print(f"{u['calls']} calls, {u['prompt_tokens'] / u['calls']:.0f} in / "
      f"{u['completion_tokens'] / u['calls']:.0f} out per call")
print(f"${u['cost_usd']:.4f} total, ${u['per_call_usd']:.6f}/call  (planned ${p['est_usd']:.2f})")
print(f"judge block {u['wall_s'] / 60:.1f} min wall at concurrency {u['concurrency']}, "
      f"{u['achieved_parallelism']:.1f}x achieved")

---
## 6 — Commit the pool, then release the GPU

In [ ]:
for path in (POOL, sidecar(POOL, 'manifest.json'), sidecar(POOL, 'usage.json')):
    print(f"{path.stat().st_size / 1e6:8.2f} MB  {path}")
print()
print(json.dumps(json.loads(sidecar(POOL, 'manifest.json').read_text())['hashes'], indent=2))

In [ ]:
# Colab only. Elsewhere the files are already on the machine that will re-rank them.
try:
    from google.colab import files

    for path in (POOL, sidecar(POOL, 'manifest.json'), sidecar(POOL, 'usage.json')):
        files.download(str(path))
except ImportError:
    print('not on Colab, nothing to download')

Stop the runtime now. The rest of the notebook makes no paid calls and touches no GPU.

---
## 7 — Rank the grid

In [ ]:
!{PY} manage.py rlsf_omega --config {CONFIG}

---
## 8 — The three readings

In [ ]:
sel = json.loads(sidecar(POOL, 'omega.json').read_text())
feature = sel['feature']

print('1. Can the reward separate a group at all?\n')
for name, s in sel['per_component_degeneracy'].items():
    print(f"   {name:8s} flat in {s['degenerate']}/{s['groups']} groups "
          f"({s['degenerate_frac']:.1%}) at N={sel['n']}")
print()
for c in sel['cells']:
    sub = c['subgroup']
    print(f"   {c['cell']:8s} combined reward flat in {c['degenerate_frac']:.1%} of groups "
          f"at N={c['n']}, {sub['degenerate_frac']:.1%} at G={sub['group_size']}")
print(f"\n   The two columns are not comparable. A group of {sel['n']} degenerates less "
      f"than a group of\n   {sel['subgroup']} by construction: more draws is more chances "
      f"for two of them to differ. G={sel['subgroup']}\n   is the figure training will see; "
      f"N={sel['n']} is what this pool could measure.")

In [ ]:
print('2. Which cell feeds Phase 2?\n')
v = sel['selection']
if v['cell']:
    print(f"   {v['cell']}: {v['reason']}")
    print(f"   set rlsf.reward to {v['weights']}")
else:
    print(f"   none: {v['reason']}")
for name, why in v['rejected'].items():
    print(f"   rejected {name}: {why}")

In [ ]:
print(f'3. Is the reward already pulling {feature}?\n')
print(f"   {'cell':10s} {'stylo_dist':>10s} {feature + ' dz':>16s}")
for c in sel['cells'] + [{'cell': k, 'picks': a} for k, a in sel['anchors'].items()]:
    s = c['picks'][f'{feature}_shift']
    print(f"   {c['cell']:10s} {c['picks']['stylo_dist']:10.3f} "
          f"{s['delta']:+9.3f} +/- {s['se']:.3f}")

g = sel['selection'].get('goodhart')
if g:
    print(f"\n   The selected cell's picks sit {g['delta']:+.2f} +/- {g['se']:.2f} above their "
          f"own groups,\n   against the {g['threshold']:.2f} band the drift rule halts a run "
          f"for.")
    if g['over_threshold']:
        print("   Best-of-N is the ceiling on what this reward can pull the policy toward, "
              "and it\n   already clears the band. Expect GRPO to trip the drift stop.")
    else:
        print("   Best-of-N is the ceiling on what this reward can pull the policy toward, "
              "and it\n   stays inside the band.")